# Step 3 — Baselines and gap diagnosis

**Purpose.** Score the standard recommenders (random, popularity, MF, NCF, NMF) under our evaluation protocol and diagnose why they fall short. This is rubric aspect 2.

**Protocol (same in every step).**
- Split by search (one search = one user), seed 4222: train / validation / test = 70 / 15 / 15%.
- Full candidate lists (median 90 products), shuffled in step 1; ties broken at random.
- NDCG@10 with gains 0 / 1 / 3 / 7 (2^label − 1); lists with no interaction are skipped.
- Every choice is made on validation. Test is scored once, in Section 6.

| Section | Content | Output (`BT4222_Group10_JDSearch/baselines/`) |
|---|---|---|
| 0 | Setup and helpers | — |
| 1 | Past project's evaluation, re-checked | table |
| 2 | Random and popularity | `score_random`, `score_popularity` |
| 3 | User × product grid from histories | kept in memory |
| 4 | MF | `score_mf`, `mf_W/H/vocab.npy` |
| 5 | NCF and NMF | `score_ncf`, `score_nmf`, `ncf.pt`, `nmf.pt` |
| 6 | Test results | `results.csv` |
| 7 | Diagnosis (validation) | `diagnosis_*.csv` |

Score files hold `sid`, `wid`, `score`, with rows in `exposures.parquet` order.

## Section 0 · Setup

- Loads `exposures.parquet` (15,510,012 rows) and the split codes.
- `ndcg_at_k` is identical to step 2. It ranks each list by score (missing scores last, ties at random) and averages NDCG@10 over lists with at least one interaction.
- `evaluate` reports NDCG@10 on train and validation. Test is held back for Section 6.
- `save_scores` stores one score per shown product. Sections 6–7 reuse them; step 4 can use them as features.



In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, gc, json, time, psutil, warnings
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import NMF

ROOT = "/content/drive/MyDrive/BT4222_Group10_JDSearch"
OUT  = f"{ROOT}/processed"                 # step-1 tables
FEAT = f"{ROOT}/features"                  # step-2 features
BASE = f"{ROOT}/baselines"                 # step-3 output: one score file per baseline
os.makedirs(BASE, exist_ok=True)

meta = json.load(open(f"{OUT}/meta.json"))
SEED, DAY = meta["seed"], 86_400
LABEL = meta["label_codes"]                # {"none": 0, "click": 1, "cart": 2, "order": 3}
SPLITS = {"train": 0, "valid": 1, "test": 2}
GAIN = np.array([0.0, 1.0, 3.0, 7.0])      # label 0/1/2/3 -> gain 2^label - 1

def ram(tag=""):
    print(f"[{tag}] RAM in use: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

def ndcg_at_k(sid, label, score, k=10, seed=0):
    """Mean NDCG@k: rank each search's list by `score` (high first, NaN last, ties broken at random)."""
    d = pd.DataFrame({"sid": sid, "label": label, "score": score})
    d["tie"] = np.random.default_rng(seed).random(len(d))
    d = d.sort_values(["sid", "score", "tie"], ascending=[True, False, True], na_position="last")
    pos = d.groupby("sid").cumcount().to_numpy()                   # 0 = top of the list
    dcg = pd.Series(np.where(pos < k, GAIN[d.label.to_numpy()] / np.log2(pos + 2), 0.0)).groupby(d.sid.to_numpy()).sum()
    best = d.sort_values(["sid", "label"], ascending=[True, False]) # the perfect order
    bpos = best.groupby("sid").cumcount().to_numpy()
    idcg = pd.Series(np.where(bpos < k, GAIN[best.label.to_numpy()] / np.log2(bpos + 2), 0.0)).groupby(best.sid.to_numpy()).sum()
    ok = idcg > 0                                                  # lists with at least one interaction
    return float((dcg[ok] / idcg[ok]).mean())

ex = pd.read_parquet(f"{OUT}/exposures.parquet")      # sid, wid, label, split
SID, LAB = ex.sid.to_numpy(), ex.label.to_numpy()
IN_SPLIT = {name: (ex.split == code).to_numpy() for name, code in SPLITS.items()}
results = []                                          # one row per baseline

def evaluate(name, score):
    """NDCG@10 on train and validation. Test is scored once, at the end (Section 6)."""
    row = {"baseline": name}
    for split in ["train", "valid"]:
        m = IN_SPLIT[split]
        row[split] = ndcg_at_k(SID[m], LAB[m], score[m])
    results.append(row)
    print(f"{name:>14s}   train {row['train']:.4f} | valid {row['valid']:.4f}")
    return row

def save_scores(name, score):
    """One score per shown product, rows in exposures order; steps 4 and 6 reuse them."""
    ex[["sid", "wid"]].assign(score=np.asarray(score, dtype="float32")).to_parquet(f"{BASE}/score_{name}.parquet", index=False)

print(f"exposures {len(ex):,} | " + " | ".join(f"{k} searches {ex.sid[m].nunique():,}" for k, m in IN_SPLIT.items()))
ram("setup")

Mounted at /content/drive
exposures 15,510,012 | train searches 120,209 | valid searches 25,759 | test searches 25,760
[setup] RAM in use: 0.8 GB


## Section 1 · Past project's evaluation, re-checked

The past project (Group 3, LSTM + hybrid NCF) reported **NDCG@10 = 0.5684** (`Models/3.1_Hybrid_NCF_with_LSTM.ipynb`, cell 94).
- **Its lists** (`Data Preparation/1. Sampling.ipynb`, cell 13): every interacted product plus min(4 × interacted, ignored) random ignored products, with interacted products stored first.
- **Its ranking** (cell 94): `np.argsort(-scores)`, with no random tie-break.

We rebuild that kind of list from our validation searches (not their exact test lists) and score three cases:

| Lists | Median size | NDCG@10 |
|---|---|---|
| Short, random order | 5 | 0.534 |
| Short, tied scores kept in stored order | 5 | 0.976 |
| Full, random order | 90 | 0.084 |

**Reading.**
- On short lists a random order already scores 0.534, so the reported 0.568 is only +0.034 above chance.
- Tied scores kept in stored order look near-perfect, because interacted products are stored first.
- Our NDCG breaks ties at random, and every result in this project uses full lists.


In [ ]:
PAST_REPORTED = 0.5684   # past project's best model (LSTM + hybrid NCF): NDCG@10 printed in 3.1_Hybrid_NCF_with_LSTM.ipynb

# We apply their rule to each of our validation lists (keep every interacted product plus 4 random ignored ones per interacted product) and score the resulting short lists. We don't use their exact test lists: those came from their own pipeline and split.
va = ex[IN_SPLIT["valid"]]
pos, neg = va[va.label > LABEL["none"]], va[va.label == LABEL["none"]]
neg = neg.assign(r=np.random.default_rng(SEED).random(len(neg))).sort_values(["sid", "r"])   # shuffle ignored products per list
n_pos = pos.groupby("sid").size()
keep = neg.groupby("sid").cumcount().to_numpy() < 4 * n_pos.reindex(neg.sid).to_numpy()     # keep 4 per interacted product
short = pd.concat([pos, neg[keep].drop(columns="r")]).sort_values("sid", kind="stable")     # interacted stored first, as in theirs

def random_ndcg(d, draws=5):
    """Average NDCG@10 of a random order over a few draws."""
    return np.mean([ndcg_at_k(d.sid.to_numpy(), d.label.to_numpy(), np.random.default_rng(i).random(len(d))) for i in range(draws)])

stored_order = -short.groupby("sid").cumcount().to_numpy()   # a "model" that ties everything and keeps the stored order
quiz = pd.DataFrame({
    "list size (median)": [short.groupby("sid").size().median()] * 2 + [va.groupby("sid").size().median()],
    "NDCG@10": [random_ndcg(short), ndcg_at_k(short.sid.to_numpy(), short.label.to_numpy(), stored_order), random_ndcg(va)]},
    index=["short lists, random order", "short lists, ties kept in stored order", "full lists, random order"])
print(quiz.round(4))
print(f"past project's reported NDCG@10 {PAST_REPORTED} = {PAST_REPORTED - quiz.iloc[0, 1]:+.3f} above random on its own kind of list")
del va, pos, neg, n_pos, keep, short, stored_order; gc.collect(); ram("section 1")


                                        list size (median)  NDCG@10
short lists, random order                              5.0   0.5341
short lists, ties kept in stored order                 5.0   0.9763
full lists, random order                              90.0   0.0840
past project's reported NDCG@10 0.5684 = +0.034 above random on its own kind of list
[section 1] RAM in use: 0.8 GB


## Section 2 · Random and popularity

- **Random:** a uniform random score. This is the floor.
- **Popularity:** step 2's label-free counts of clicks (7 days), carts (90 days) and orders (1 year) on the product, across all users' histories, using events at least 1 day old. Every user gets the same order. The best variant is chosen on validation.


| Baseline | Train | Validation |
|---|---|---|
| random | 0.0809 | 0.0815 |
| pop_click_7d | 0.3192 | 0.3198 |
| **pop_cart_90d** | 0.3326 | **0.3342** |
| pop_order_1y | 0.2873 | 0.2905 |

**Reading.** Cart popularity is the popularity baseline at 0.3342, about 4× random. Train and validation are close, since nothing is fitted.


In [ ]:
random_score = np.random.default_rng(SEED).random(len(ex))
evaluate("random", random_score)
save_scores("random", random_score)

# Popularity = step 2's label-free counts (all users' history, events >= 1 day old). Nothing new is computed.
pop = pd.read_parquet(f"{FEAT}/section3_popularity.parquet", columns=["sid", "wid", "pop_click_7d", "pop_cart_90d", "pop_order_1y"])
assert (pop.sid.to_numpy() == SID).all() and (pop.wid.to_numpy() == ex.wid.to_numpy()).all()   # rows line up with ex
pop_valid = {}
for c in ["pop_click_7d", "pop_cart_90d", "pop_order_1y"]:
    pop_valid[c] = evaluate(c, pop[c].to_numpy())["valid"]
best_pop = max(pop_valid, key=pop_valid.get)                   # chosen on validation only
save_scores("popularity", pop[best_pop].to_numpy())
print("popularity baseline =", best_pop, "(best on validation)")
del pop, random_score; gc.collect(); ram("section 2")

        random   train 0.0809 | valid 0.0815
  pop_click_7d   train 0.3192 | valid 0.3198
  pop_cart_90d   train 0.3326 | valid 0.3342
  pop_order_1y   train 0.2873 | valid 0.2905
popularity baseline = pop_cart_90d (best on validation)
[section 2] RAM in use: 0.7 GB


## Section 3 · User × product grid

This is the standard collaborative-filtering input.
- **Rows:** 173,831 users (one per search).
- **Columns:** products.
- **Cell:** log(1 + number of actions on that product). All action types count equally, as in the standard form.

Choices:
- **Which products get a column:** those with at least 2 distinct users, since a product touched by one person links no one.
- **`X_old`** (events at least 1 day old, all users) learns product vectors. This is the same anti-leak rule as step 2's popularity: another user's same-day event may come after the search.
- **`X_all`** (each user's full own history) describes the user. It uses only that user's actions before the search, and no labels.

**Result.**
- 1,545,912 of 5,326,956 history products get a column, and the grid is 0.0049% filled.
- Shown products with a column: 31.8% of shown rows and 61.1% of interacted rows. All other shown products are cold for MF, NCF and NMF.


In [ ]:
MIN_USERS = 2                                   # a product needs >= 2 different users to link people's tastes
N_USERS = meta["table_rows"]["searches"]        # one row per search = one user (sid 0 ... 173,830)
hist = pd.read_parquet(f"{OUT}/history.parquet", columns=["sid", "wid", "age_s"])
old = (hist.age_s >= DAY).to_numpy()            # pre-17-Oct events: safe to pool across users (same rule as step 2, Section 3)
users_per_product = hist[old].drop_duplicates(["sid", "wid"]).wid.value_counts()
vocab = np.sort(users_per_product.index[users_per_product >= MIN_USERS].to_numpy())    # products that get a column

def history_matrix(rows):
    """User x product grid from the chosen history rows: cell = log(1 + number of actions). Products without a column are skipped."""
    wid = hist.wid.to_numpy()[rows]
    col = np.minimum(np.searchsorted(vocab, wid), len(vocab) - 1)    # column of each product, if it has one
    ok = vocab[col] == wid                                            # True = product has a column
    X = sparse.csr_matrix((np.ones(ok.sum(), np.float32), (hist.sid.to_numpy()[rows][ok], col[ok])), shape=(N_USERS, len(vocab)))
    X.sum_duplicates()                                                # repeat actions on one product add up
    X.data = np.log1p(X.data)                                         # 1 action -> 0.69, 10 -> 2.40: tames heavy repeaters
    return X

X_old = history_matrix(old)                                 # to learn product tastes
X_all = history_matrix(np.ones(len(hist), dtype=bool))      # each user's full own history, to describe the user

col = np.minimum(np.searchsorted(vocab, ex.wid.to_numpy()), len(vocab) - 1)
warm = vocab[col] == ex.wid.to_numpy()                      # shown product has a column ("warm")
print(f"products with a column: {len(vocab):,} of {len(users_per_product):,} | filled cells: {X_old.nnz:,} old, {X_all.nnz:,} all "
      f"| filled share {X_all.nnz / (N_USERS * len(vocab)):.6%}")
print(f"shown products with a column: {warm.mean():.1%} of rows, {warm[LAB > LABEL['none']].mean():.1%} of interacted rows")
del hist, old, users_per_product; gc.collect(); ram("section 3")

products with a column: 1,545,912 of 5,326,956 | filled cells: 12,642,956 old, 13,189,435 all | filled share 0.004908%
shown products with a column: 31.8% of rows, 61.1% of interacted rows
[section 3] RAM in use: 1.7 GB


## Section 4 · MF

Matrix factorisation: approximates the grid as W × H with 32 hidden tastes; score = user tastes · product tastes.
- Fitted with sklearn's `NMF` class, which keeps every number ≥ 0.
- `fit(X_old)` learns H: each product's mix of tastes, from events before 17 Oct.
- `transform(X_all)` holds H fixed and solves W: each user's mix, from their own full history. No labels are used.
- **Score** = W[user] · H[:, product]. A cold product has no column, so it gets no score and is ranked last.
- **Rounds:** 100, to keep the Colab runtime short.

**Result.**
- Train 0.3158, validation 0.3159: below cart popularity (0.3342), although MF is personalised.
- 0.8% of users get an all-zero taste vector.


In [ ]:
N_FACTORS, MF_ROUNDS = 32, 100       # 32 hidden tastes; 100 rounds
warnings.filterwarnings("ignore", message="Maximum number of iterations")   # stopping at the round limit is expected
t = time.time()
mf = NMF(n_components=N_FACTORS, init="nndsvda", max_iter=MF_ROUNDS, random_state=SEED)
mf.fit(X_old)                                      # learns product tastes from pooled pre-17-Oct histories
H = mf.components_.astype("float32")               # 32 x products
W = mf.transform(X_all).astype("float32")          # each user's tastes from their own full history (products held fixed)
print(f"MF: {mf.n_iter_} rounds in {(time.time() - t) / 60:.1f} min | users with an empty taste vector: {(W.sum(axis=1) == 0).mean():.1%}")
np.save(f"{BASE}/mf_W.npy", W); np.save(f"{BASE}/mf_H.npy", H); np.save(f"{BASE}/mf_vocab.npy", vocab)

mf_score = np.full(len(ex), np.nan, dtype=np.float32)       # cold product: no column, no score (ranked last)
HT = np.ascontiguousarray(H.T)                                # one row of 32 tastes per product
for a in range(0, len(ex), 2_000_000):                        # 2M rows at a time keeps memory low
    rows = a + np.flatnonzero(warm[a:a + 2_000_000])
    mf_score[rows] = (W[SID[rows]] * HT[col[rows]]).sum(axis=1)   # match = sum over tastes of user x product
evaluate("mf", mf_score)
save_scores("mf", mf_score)
del HT, rows, mf_score; gc.collect(); ram("section 4")
print(pd.DataFrame(results).set_index("baseline").round(4))

MF: 100 rounds in 7.2 min | users with an empty taste vector: 0.8%
            mf   train 0.3158 | valid 0.3159
[section 4] RAM in use: 1.7 GB
               train   valid
baseline                    
random        0.0809  0.0815
pop_click_7d  0.3192  0.3198
pop_cart_90d  0.3326  0.3342
pop_order_1y  0.2873  0.2905
mf            0.3158  0.3159


## Section 5 · NCF and NMF

The two neural CF models built from one class:
- **NCF:** user and product vectors are joined and passed through a small network (64 → 32 → 16) to a score.
- **NMF (Neural Matrix Factorization):** NCF plus a GMF branch (user × product, number by number, as in MF). A final layer combines both.

**Training (same for both).**
- Each product a user touched is a real pair; each gets 4 random products as fake pairs. The model learns to tell them apart.
- *Fit:* products, network and users learn from events before 17 Oct (`X_old`).
- *Fold-in:* products and network frozen; only user vectors learn, now from each user's full history (`X_all`).
- Validation is scored before training and after every epoch, and the best weights are kept, so a stage that doesn't help is undone.
- Cold products get no score and are ranked last.

| Stage | NCF valid | NMF valid |
|---|---|---|
| Start (untrained) | 0.1766 | 0.1771 |
| Fit, epoch 1 | 0.3243 | 0.3249 |
| Fit, epoch 2 | 0.3269 | **0.3260** (kept) |
| Fit, epoch 3 | 0.3256 | 0.3186 |
| Fold-in | **0.3271** (kept) | 0.3230 (undone) |

| Baseline | Train | Valid |
|---|---|---|
| MF | 0.3158 | 0.3159 |
| **NCF** | 0.3261 | **0.3271** |
| NMF | 0.3256 | 0.3260 |
| Popularity (carts, 90 days) | 0.3326 | 0.3342 |

**Reading.**
- NCF and NMF beat MF but still trail cart popularity. Adding the GMF branch (NMF) does not beat NCF.
- Validation peaks at epoch 2 for both while the loss keeps falling (NMF: 0.39 → 0.25). Fitting past history better doesn't rank search results better.
- Fold-in barely matters: +0.0002 for NCF (kept), −0.003 for NMF (undone).
- An untrained model already scores 0.177, about twice random, just by ranking cold products last.

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
DEV = "cuda"

N_DIM, N_NEG = 32, 4             # vector size (same as MF); random products per real pair
BATCH, LR = 4096, 1e-3           # real pairs per training step; Adam learning rate
FIT_EPOCHS, FOLD_EPOCHS = 3, 1   # the best epoch on validation is kept

class NeuralCF(nn.Module):
    """NCF: user + product vectors -> MLP -> score.
    use_gmf=True adds a GMF branch (user x product, element-wise), making it NMF (Neural Matrix Factorization)."""
    def __init__(self, n_users, n_items, d, use_gmf):
        super().__init__()
        self.use_gmf = use_gmf
        self.mlp_user, self.mlp_item = nn.Embedding(n_users, d), nn.Embedding(n_items, d)
        tables = [self.mlp_user, self.mlp_item]
        if use_gmf:
            self.gmf_user, self.gmf_item = nn.Embedding(n_users, d), nn.Embedding(n_items, d)
            tables += [self.gmf_user, self.gmf_item]
        for table in tables:
            nn.init.normal_(table.weight, std=0.01)
        self.mlp = nn.Sequential(nn.Linear(2 * d, d), nn.ReLU(), nn.Linear(d, d // 2), nn.ReLU())
        self.out = nn.Linear(d // 2 + (d if use_gmf else 0), 1)

    def forward(self, user, item):
        parts = [self.mlp(torch.cat([self.mlp_user(user), self.mlp_item(item)], dim=1))]
        if self.use_gmf:
            parts.append(self.gmf_user(user) * self.gmf_item(item))
        return self.out(torch.cat(parts, dim=1)).squeeze(1)       # higher = more likely a real pair

@torch.no_grad()
def cf_scores(model):
    """Score every shown product that has a column; cold products stay NaN (ranked last)."""
    score = np.full(len(ex), np.nan, dtype=np.float32)
    rows = np.flatnonzero(warm)
    for a in range(0, len(rows), 1_000_000):
        r = rows[a:a + 1_000_000]
        u = torch.from_numpy(SID[r].astype(np.int64)).to(DEV)
        i = torch.from_numpy(col[r]).to(DEV)
        score[r] = model(u, i).cpu().numpy()
    return score

def train(model, X, params, epochs, tag):
    """Real pairs = filled cells of X (label 1), each with N_NEG random products (label 0), redrawn every epoch.
    Scores validation before training and after each epoch, and keeps the best weights (a stage that doesn't help is undone)."""
    X = X.tocoo()
    users = torch.from_numpy(X.row.astype(np.int64)).to(DEV)
    items = torch.from_numpy(X.col.astype(np.int64)).to(DEV)
    opt = torch.optim.Adam(params, lr=LR)
    m = IN_SPLIT["valid"]
    best_valid = ndcg_at_k(SID[m], LAB[m], cf_scores(model)[m])
    best_state = copy.deepcopy(model.state_dict())
    print(f"[{tag}] start: valid NDCG@10 {best_valid:.4f}")
    for epoch in range(epochs):
        t, total = time.time(), 0.0
        order = torch.randperm(len(users), device=DEV)
        for a in range(0, len(users), BATCH):
            b = order[a:a + BATCH]
            u = users[b].repeat(1 + N_NEG)                                   # each user: 1 real + N_NEG fake
            i = torch.cat([items[b], torch.randint(0, X.shape[1], (len(b) * N_NEG,), device=DEV)])
            y = torch.zeros(len(u), device=DEV)
            y[:len(b)] = 1.0
            loss = F.binary_cross_entropy_with_logits(model(u, i), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.detach() * len(b)
        valid = ndcg_at_k(SID[m], LAB[m], cf_scores(model)[m])
        print(f"[{tag}] epoch {epoch + 1}/{epochs}: loss {total.item() / len(users):.4f} | valid NDCG@10 {valid:.4f} | {time.time() - t:.0f}s")
        if valid > best_valid:
            best_valid, best_state = valid, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)

for name, use_gmf in [("ncf", False), ("nmf", True)]:
    t = time.time()
    torch.manual_seed(SEED)
    model = NeuralCF(N_USERS, len(vocab), N_DIM, use_gmf).to(DEV)
    train(model, X_old, model.parameters(), FIT_EPOCHS, f"{name} fit")       # products, network, users: events before 17 Oct

    user_vectors = [model.mlp_user.weight] + ([model.gmf_user.weight] if use_gmf else [])
    for p in model.parameters():                                              # fold-in: freeze everything ...
        p.requires_grad_(False)
    for p in user_vectors:                                                    # ... except the user vectors
        p.requires_grad_(True)
    train(model, X_all, user_vectors, FOLD_EPOCHS, f"{name} fold-in")        # each user's full own history
    print(f"{name} trained in {(time.time() - t) / 60:.1f} min")
    torch.save(model.state_dict(), f"{BASE}/{name}.pt")

    s = cf_scores(model)
    evaluate(name, s)
    save_scores(name, s)
    del model, s; gc.collect()

ram("section 5")
print(pd.DataFrame(results).set_index("baseline").round(4))

[ncf fit] start: valid NDCG@10 0.1766
[ncf fit] epoch 1/3: loss 0.3931 | valid NDCG@10 0.3243 | 60s
[ncf fit] epoch 2/3: loss 0.3639 | valid NDCG@10 0.3269 | 60s
[ncf fit] epoch 3/3: loss 0.3451 | valid NDCG@10 0.3256 | 60s
[ncf fold-in] start: valid NDCG@10 0.3269
[ncf fold-in] epoch 1/1: loss 0.3449 | valid NDCG@10 0.3271 | 11s
ncf trained in 3.4 min
           ncf   train 0.3261 | valid 0.3271
[nmf fit] start: valid NDCG@10 0.1771
[nmf fit] epoch 1/3: loss 0.3913 | valid NDCG@10 0.3249 | 117s
[nmf fit] epoch 2/3: loss 0.3345 | valid NDCG@10 0.3260 | 117s
[nmf fit] epoch 3/3: loss 0.2538 | valid NDCG@10 0.3186 | 117s
[nmf fold-in] start: valid NDCG@10 0.3260
[nmf fold-in] epoch 1/1: loss 0.2594 | valid NDCG@10 0.3230 | 18s
nmf trained in 6.2 min
           nmf   train 0.3256 | valid 0.3260
[section 5] RAM in use: 2.8 GB
               train   valid
baseline                    
random        0.0809  0.0815
pop_click_7d  0.3192  0.3198
pop_cart_90d  0.3326  0.3342
pop_order_1y  0.2873 

## Section 6 · Test results

Each baseline's saved scores are scored once on the test set (25,760 searches). Nothing is chosen after this.

| Baseline | Train | Valid | Test |
|---|---|---|---|
| random | 0.0809 | 0.0815 | 0.0804 |
| **popularity** (carts, 90 days) | 0.3326 | 0.3342 | **0.3329** |
| MF | 0.3158 | 0.3159 | 0.3168 |
| NCF | 0.3261 | 0.3271 | 0.3290 |
| NMF | 0.3256 | 0.3260 | 0.3267 |

**Reading.**
- Test is within 0.002 of validation for every baseline, so the choices made on validation hold up.
- Plain popularity is the best baseline. All three CF models lose to it: NCF by 0.004, NMF by 0.006, MF by 0.016.

In [ ]:
FINAL = ["random", "popularity", "mf", "ncf", "nmf"]
table = pd.DataFrame(results).set_index("baseline").rename(index={best_pop: "popularity"})

m = IN_SPLIT["test"]
test = {}
for name in FINAL:                                           # score the saved files, so test uses exactly what was saved
    s = pd.read_parquet(f"{BASE}/score_{name}.parquet")
    assert (s.sid.to_numpy() == SID).all() and (s.wid.to_numpy() == ex.wid.to_numpy()).all()
    test[name] = ndcg_at_k(SID[m], LAB[m], s.score.to_numpy()[m])

final = table.loc[FINAL, ["train", "valid"]].assign(test=pd.Series(test))
final.to_csv(f"{BASE}/results.csv")
print(f"test searches: {ex.sid[m].nunique():,}")
final.round(4)

test searches: 25,760


,train,valid,test
baseline,,,
random,0.0809,0.0815,0.0804
popularity,0.3326,0.3342,0.3329
mf,0.3158,0.3159,0.3168
ncf,0.3261,0.3271,0.3290
nmf,0.3256,0.3260,0.3267


## Section 7 · Diagnosis

Why do MF, NCF and NMF lose to popularity? Each test splits the validation searches into groups and compares the baselines. Validation is used because these results guide steps 4–5; test stays untouched.

### 7.1 Cold products
Searches grouped by how many of their interacted products have a column in the grid.

| Interacted products with a column | Searches | Random | Popularity | MF | NCF | NMF |
|---|---|---|---|---|---|---|
| All | 15,396 | 0.0728 | 0.4567 | 0.4394 | 0.4565 | 0.4545 |
| Some | 3,785 | 0.0751 | 0.2376 | 0.2327 | 0.2401 | 0.2407 |
| None | 6,578 | 0.1054 | 0.1012 | 0.0737 | 0.0737 | 0.0737 |

- In about a quarter of searches (6,578 of 25,759), CF can't score any interacted product. It ranks them last and falls below random.
- Popularity fails there too (0.101, below random): these searches end on niche products few users have touched.
- When CF can score the interacted products (All, Some), NCF and NMF are within 0.003 of popularity. Personalisation adds almost nothing.

### 7.2 Same blind spot
Popularity re-scored with cold products ranked last, like CF.

| Score | Valid NDCG@10 |
|---|---|
| Popularity (all products) | 0.3342 |
| Popularity, cold products last | 0.3303 |
| MF | 0.3159 |
| NCF | 0.3271 |
| NMF | 0.3260 |

Rank correlation with popularity within each list: MF 0.650, NCF 0.706, NMF 0.690.

- The blind spot costs popularity only 0.004, and it still beats all three CF models.
- CF's ranking largely copies popularity. Random products as negatives teach "popular vs obscure", not personal taste; Section 5 shows validation stalling while the loss keeps falling.

### 7.3 User activity (clicks in the last 7 days)

| Clicks | Searches | Random | Popularity | MF | NCF | NMF |
|---|---|---|---|---|---|---|
| 0 | 4,184 | 0.0798 | 0.3468 | 0.3147 | 0.3347 | 0.3337 |
| 1–20 | 9,648 | 0.0800 | 0.3336 | 0.3159 | 0.3291 | 0.3286 |
| 21–100 | 8,605 | 0.0827 | 0.3321 | 0.3188 | 0.3253 | 0.3245 |
| Over 100 | 3,322 | 0.0846 | 0.3206 | 0.3067 | 0.3126 | 0.3089 |

- All three CF models trail popularity at every activity level, even for users with over 100 clicks. MF's gap narrows from 0.032 to about 0.014 but never closes.
- Popularity, NCF and NMF all score lower for heavier users, whose interacted products are less often in the grid (63% → 58%).

### 7.4 New vs repeat search
Repeat = the user acted on results of a similar search in the last 7 days.

| Search | Searches | Random | Popularity | MF | NCF | NMF |
|---|---|---|---|---|---|---|
| New | 22,496 | 0.0804 | 0.3386 | 0.3182 | 0.3313 | 0.3297 |
| Repeat | 3,263 | 0.0892 | 0.3032 | 0.3004 | 0.2978 | 0.3004 |

- Popularity and all three CF models score lower on repeat searches (popularity 0.339 → 0.303).
- The user just acted on similar results, yet no CF model beats popularity there: CF never sees the query.

### Summary

| Weakness | Evidence | Answer in steps 4–5 |
|---|---|---|
| Cold products | Searches with no interacted product in the grid: CF 0.074, popularity 0.101, random 0.105 | Product attributes, title–query match |
| Learns popularity, not taste | Rank correlation 0.65–0.71; loses to popularity with the same blind spot | Learn from shown-but-ignored products |
| Doesn't use history well | Trails popularity at every activity level | Separate click / cart / order streams with recency |
| Ignores the query | Repeat searches: all CF below popularity | Query–title and query-matched features |

In [ ]:
m = IN_SPLIT["valid"]                      # diagnosis guides steps 4-5, so it uses validation; test stays untouched
POS = LAB > LABEL["none"]                  # interacted rows
NAMES = ["random", "popularity", "mf", "ncf", "nmf"]
CF = ["mf", "ncf", "nmf"]
scores = {name: pd.read_parquet(f"{BASE}/score_{name}.parquet").score.to_numpy() for name in NAMES}

def by_segment(segment, order):
    """Validation NDCG@10 per segment. `segment` maps each search id to a segment name."""
    seg = segment.reindex(SID).to_numpy()  # segment name on every row
    rows = []
    for g in order:
        s = m & (seg == g)
        row = {"segment": g, "searches": len(np.unique(SID[s])), "interacted with a column": warm[s & POS].mean()}
        for name in NAMES:
            row[name] = ndcg_at_k(SID[s], LAB[s], scores[name][s])
        rows.append(row)
    return pd.DataFrame(rows).set_index("segment").round(4)

# 7.1 Cold products: how many of each search's interacted products have a column?
share = pd.Series(warm[m & POS]).groupby(SID[m & POS]).mean()
cold = pd.Series(np.select([share == 1, share == 0], ["all have a column", "none has a column"], "some have a column"), index=share.index)
t71 = by_segment(cold, ["all have a column", "some have a column", "none has a column"])
print("7.1 Cold products"); display(t71)

# 7.2 Same blind spot: popularity with cold products ranked last, like CF
blind_spot = {"popularity (all products)": scores["popularity"],
              "popularity, cold products last": np.where(warm, scores["popularity"], np.nan),
              "mf": scores["mf"], "ncf": scores["ncf"], "nmf": scores["nmf"]}
t72 = pd.Series({k: ndcg_at_k(SID[m], LAB[m], s[m]) for k, s in blind_spot.items()}, name="valid NDCG@10").round(4)
print("7.2 Same blind spot"); display(t72)

warnings.filterwarnings("ignore", message="An input array is constant")   # lists where every product ties
w = m & warm                                                              # products CF can score
d = pd.DataFrame({name: scores[name][w] for name in ["popularity"] + CF}).assign(sid=SID[w])
follow = {name: d.groupby("sid")[[name, "popularity"]].apply(lambda x: x[name].corr(x.popularity, method="spearman")).mean()
          for name in CF}                                                 # rank correlation within each list, averaged
print("rank correlation with popularity, within lists:", " | ".join(f"{name} {v:.3f}" for name, v in follow.items()))

# 7.3 User activity: clicks in the last 7 days (step 2, Section 4)
ctx = pd.read_parquet(f"{FEAT}/section4_context.parquet").set_index("sid")
activity = pd.cut(ctx.clicks_7d, [-1, 0, 20, 100, np.inf], labels=["0", "1-20", "21-100", "over 100"]).astype(str)
t73 = by_segment(activity, ["0", "1-20", "21-100", "over 100"])
print("7.3 Clicks in the last 7 days"); display(t73)

# 7.4 New vs repeat search: step 2's query-matched feature exists only if the user acted on a similar search in the last 7 days
qm = pd.read_parquet(f"{FEAT}/section2_affinity.parquet", columns=["sid", "qmatch_cate4_share_7d"])
assert (qm.sid.to_numpy() == SID).all()
repeat = qm.qmatch_cate4_share_7d.notna().groupby(qm.sid).any().map({True: "repeat", False: "new"})
t74 = by_segment(repeat, ["new", "repeat"])
print("7.4 New vs repeat search"); display(t74)

for name, t in [("cold", t71), ("blind_spot", t72), ("activity", t73), ("query", t74)]:
    t.to_csv(f"{BASE}/diagnosis_{name}.csv")
del scores, blind_spot, d, qm, ctx; gc.collect(); ram("section 7")

7.1 Cold products


,searches,interacted with a column,random,popularity,mf,ncf,nmf
segment,,,,,,,
all have a column,15396,1.0000,0.0728,0.4567,0.4394,0.4565,0.4545
some have a column,3785,0.4951,0.0751,0.2376,0.2327,0.2401,0.2407
none has a column,6578,0.0000,0.1054,0.1012,0.0737,0.0737,0.0737


7.2 Same blind spot


,valid NDCG@10
popularity (all products),0.3342
"popularity, cold products last",0.3303
mf,0.3159
ncf,0.3271
nmf,0.3260


rank correlation with popularity, within lists: mf 0.650 | ncf 0.706 | nmf 0.690
7.3 Clicks in the last 7 days


,searches,interacted with a column,random,popularity,mf,ncf,nmf
segment,,,,,,,
0,4184,0.6304,0.0798,0.3468,0.3147,0.3347,0.3337
1-20,9648,0.6069,0.0800,0.3336,0.3159,0.3291,0.3286
21-100,8605,0.6145,0.0827,0.3321,0.3188,0.3253,0.3245
over 100,3322,0.5846,0.0846,0.3206,0.3067,0.3126,0.3089


7.4 New vs repeat search


,searches,interacted with a column,random,popularity,mf,ncf,nmf
segment,,,,,,,
new,22496,0.6199,0.0804,0.3386,0.3182,0.3313,0.3297
repeat,3263,0.5500,0.0892,0.3032,0.3004,0.2978,0.3004


[section 7] RAM in use: 3.3 GB
